# Working with larger-than-memory data

Every chapter so far loaded data fully into memory. Real datasets don't always
fit. `polars`' **lazy** and **streaming** execution let you build a pipeline
that never materializes the whole thing at once.

```{note}
We demonstrate on the small [ETL dataset](../01c-etl/data-preparation.ipynb) for
practicality — but the *technique* is what scales, not the demo data. The same
code runs unchanged on a file far too big for RAM.
```

## The lazy API: build a plan, don't run it

`LazyCsvReader` *scans* rather than reads — it records what you want without
touching the data. Chaining `filter`/`select` builds a **query plan**; nothing
executes until `.collect()`.

In [ ]:
:dep polars = { version = "0.44", features = ["lazy", "csv", "streaming"] }
use polars::prelude::*;

let query: LazyFrame = LazyCsvReader::new("/book/data/customers.csv")
    .with_has_header(true)
    .finish()?
    .filter(col("churned").eq(lit(1)))
    .select([col("city"), col("income")]);
println!("query plan built — no data read yet");

## Query optimization: pushdown

Before running, polars *optimizes* the plan. `explain(true)` shows the optimized
version — note how the filter and the column selection are **pushed down** to the
CSV scan, so only the needed rows and columns are ever read from disk:

In [ ]:
println!("{}", query.clone().explain(true)?);

That pushdown is the whole game: on a 100 GB file, reading only two columns of
the matching rows is the difference between feasible and impossible.

## Streaming execution

`with_streaming(true)` runs the query in **chunks** that fit in memory, rather
than loading the full input. The result is identical; the memory profile isn't:

In [ ]:
{
    let result = query.clone().with_streaming(true).collect()?;
    println!("streamed result: {} rows x {} cols", result.height(), result.width());
    println!("{}", result);
}

## Where this stops: training

```{warning}
Be honest about the boundary: the modelling crates in this book (`linfa`,
`smartcore`) expect an **in-memory** `ndarray`/`DenseMatrix`. So "larger than
memory" in the Rust ecosystem today applies to the **ETL / feature-engineering**
stage — cleaning, filtering, aggregating with lazy + streaming polars — not to
model *training* itself, which still needs its inputs in RAM. Out-of-core
training is not a solved story here.
```

The practical pattern: use lazy + streaming polars to reduce a huge raw dataset
down to the aggregated, filtered feature matrix that *does* fit, then train on
that. It closes the loop on the [ETL chapter](../01c-etl/data-preparation.ipynb),
now at scale.

Next: the [Capstone](../12-capstone/end-to-end-project.ipynb) — every chapter's
technique on one dataset, start to finish.